# NB01 · 데이터 탐색 (EDA)

| | |
|---|---|
| **Author** | 한의정 (Data Engineering Lead) |
| **Description** | 원본 데이터셋의 구조와 분포를 파악하여 전처리 전략 수립의 근거를 마련합니다. |
| **Input** | `merged_annotations_train_final.json`, `train_images/` |
| **Output** | 분포 시각화 (클래스, 이미지 크기, BBox 크기, 객체 수, 픽셀 레벨 분석) |
| **Next** | NB02 - Stratified Split + Copy-Paste 증강 |


In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║                  ⚠️  시작 전 필독 공지  ⚠️                       ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  [한글 폰트 설치 - Colab 최초 1회]                               ║
║   !apt-get install -y fonts-nanum > /dev/null               ║
║   실행 후 런타임 재시작 필요                                       ║
║                                                              ║
║  [데이터 경로]                                                  ║
║   구글 드라이브 /MyDrive/data/초급_프로젝트/dataset/                ║
║   → 원본 데이터(merged_annotations_train_final.json) 필수!       ║
║                                                              ║
║  [전처리 순서]                                                  ║
║   NB01(EDA) → NB02 → NB03 → NB04 → NB05                     ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")


In [ ]:
import sys, os

# Colab 감지: /content/drive 존재 여부로 판단 (get_ipython 불필요)
is_colab = os.path.exists('/content/drive')

if is_colab:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception:
        pass  # 이미 마운트됐거나 VSCode 커널 환경

    REPO_DIR = '/content/pill_detection_project'
    if not os.path.exists(REPO_DIR):
        os.system('git clone https://github.com/wina0901/pill_detection_project.git ' + REPO_DIR)

    # nested 폴더 대응 (pill_detection_project/pill_detection_project/src/ 구조)
    PROJECT_ROOT = REPO_DIR
    nested = os.path.join(REPO_DIR, 'pill_detection_project')
    if os.path.isdir(os.path.join(nested, 'src')):
        PROJECT_ROOT = nested

    sys.path.insert(0, PROJECT_ROOT)
    BASE_DIR = '/content/drive/MyDrive/data/초급_프로젝트/dataset'

else:
    REPO_DIR = os.path.expanduser('~/Desktop/vera_pill')
    sys.path.insert(0, REPO_DIR)
    BASE_DIR = os.path.expanduser('~/Desktop/vera_pill/dataset')

print(f"환경    : {'Colab' if is_colab else '로컬'}")
print(f"REPO_DIR: {sys.path[0]}")
print(f"BASE_DIR: {BASE_DIR}")
assert os.path.exists(BASE_DIR), f"🚨 경로 없음: {BASE_DIR}"


## 0. 라이브러리 로드

In [ ]:
import os, json, platform, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import cv2
import glob
import torch
from IPython.display import display
warnings.filterwarnings('ignore')

# OS별 한글 폰트 설정
if platform.system() == 'Darwin':
    plt.rc('font', family='AppleGothic')
    plt.rcParams['axes.unicode_minus'] = False
    fp = fm.FontProperties(family='AppleGothic')
    print("🍎 macOS: AppleGothic 폰트 적용")
else:
    font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    if os.path.exists(font_path):
        fp = fm.FontProperties(fname=font_path)
        plt.rc('font', family=fp.get_name())
        print("🐧 Linux(Colab): NanumGothic 폰트 적용")
    else:
        fp = fm.FontProperties()
        print("⚠️  한글 폰트 없음 → !apt-get install -y fonts-nanum 실행 후 재시작")

# GPU/MPS 장치 확인
def get_device():
    if torch.backends.mps.is_available():
        return torch.device('mps'), 'Apple Silicon MPS'
    elif torch.cuda.is_available():
        return torch.device('cuda'), f'NVIDIA GPU ({torch.cuda.get_device_name(0)})'
    return torch.device('cpu'), 'CPU'

DEVICE, device_name = get_device()
print(f"✅ DEVICE: {DEVICE}  [{device_name}]")
print("✅ 라이브러리 로드 완료")


## 1. 데이터 로드 및 구조 확인

In [ ]:
TRAIN_JSON = os.path.join(BASE_DIR, 'merged_annotations_train_final.json')
assert os.path.exists(TRAIN_JSON), f"🚨 파일 없음: {TRAIN_JSON}"

with open(TRAIN_JSON, 'r', encoding='utf-8') as f:
    coco_data = json.load(f)

images_df      = pd.DataFrame(coco_data['images'])
categories_df  = pd.DataFrame(coco_data['categories'])
annotations_df = pd.DataFrame(coco_data['annotations'])

cat_dict = dict(zip(categories_df['id'], categories_df['name']))
annotations_df['class_name'] = annotations_df['category_id'].map(cat_dict)
annotations_df[['x', 'y', 'w', 'h']] = pd.DataFrame(
    annotations_df['bbox'].tolist(), index=annotations_df.index)
annotations_df['area']         = annotations_df['w'] * annotations_df['h']
annotations_df['aspect_ratio'] = annotations_df['w'] / annotations_df['h']
annotations_df['center_x']    = annotations_df['x'] + annotations_df['w'] / 2
annotations_df['center_y']    = annotations_df['y'] + annotations_df['h'] / 2

print(f"{'='*20} 데이터셋 규모 {'='*20}")
print(f"• 총 이미지 수        : {len(images_df):,} 장")
print(f"• 알약 클래스 수      : {len(categories_df):,} 종")
print(f"• Bounding Box 수     : {len(annotations_df):,} 개")
print(f"• 이미지당 평균 객체  : {len(annotations_df)/len(images_df):.2f} 개")
print()

# 해상도 분포
res_counts = images_df.groupby(['width', 'height']).size().reset_index(name='count')
print(f"{'='*20} 이미지 해상도 분포 {'='*20}")
print(res_counts.to_string(index=False))
print()

# 결측치 검사
print(f"{'='*20} 결측치 검사 {'='*20}")
null_report = annotations_df.isnull().sum()
print(null_report[null_report > 0] if null_report.sum() > 0 else "결측치 없음 ✅")
print()

# 클래스 불균형
class_counts = annotations_df['class_name'].value_counts()
print(f"{'='*20} 클래스 불균형 {'='*20}")
print(f"[최다 5개]"); print(class_counts.head(5))
print(f"\n[최소 5개]"); print(class_counts.tail(5))
print()

# BBox 크기
min_area, max_area = annotations_df['area'].min(), annotations_df['area'].max()
print(f"{'='*20} 객체 크기(Scale) {'='*20}")
print(f"• 최소 면적: {min_area:,.0f} px²")
print(f"• 최대 면적: {max_area:,.0f} px²")
print(f"• 면적 Ratio: {max_area/min_area:.1f}배  → Letterbox 필요 근거")


## 2. 마스터 분석 대시보드

In [ ]:
pills_per_image = annotations_df.groupby('image_id').size()

fig = plt.figure(figsize=(25, 15))
plt.subplots_adjust(hspace=0.35, wspace=0.25)
fig.suptitle('■ [Health Eat] 데이터셋 마스터 분석 대시보드',
             fontproperties=fp, fontsize=26, fontweight='bold', y=0.97)

# [1] 클래스 불균형 Top10 & Bottom10
ax1 = plt.subplot(2, 3, 1)
extreme = pd.concat([class_counts.head(10), class_counts.tail(10)])
sns.barplot(x=extreme.values, y=extreme.index, palette='coolwarm', ax=ax1)
ax1.set_title('[!] 클래스 불균형 (Top 10 & Bottom 10)', fontproperties=fp, fontsize=15)
ax1.set_xlabel('객체 수')
for label in ax1.get_yticklabels():
    label.set_fontproperties(fp)
    label.set_fontsize(8)

# [2] 객체 위치 히트맵 (Spatial Distribution)
ax2 = plt.subplot(2, 3, 2)
sns.kdeplot(x=annotations_df['center_x'], y=annotations_df['center_y'],
            fill=True, cmap='Reds', ax=ax2)
ax2.set_xlim(0, images_df['width'].max())
ax2.set_ylim(images_df['height'].max(), 0)
ax2.set_title('[+] 알약 배치 위치 히트맵', fontproperties=fp, fontsize=15)
ax2.set_xlabel('X 좌표'); ax2.set_ylabel('Y 좌표')

# [3] 가로세로 비율 (Aspect Ratio)
ax3 = plt.subplot(2, 3, 3)
sns.histplot(annotations_df['aspect_ratio'], bins=30, kde=True, color='teal', ax=ax3)
ax3.axvline(1.0, color='red', linestyle='--', label='정사각형 기준')
ax3.set_title('[+] BBox 가로세로 비율 (W/H)', fontproperties=fp, fontsize=15)
ax3.legend(prop=fp)

# [4] 알약 면적 분포
ax4 = plt.subplot(2, 3, 4)
sns.histplot(annotations_df['area'], bins=50, kde=True, color='purple', ax=ax4)
ax4.set_title('[+] 알약 면적(Area) 분포', fontproperties=fp, fontsize=15)
ax4.set_xlabel('면적 (px²)')

# [5] 이미지당 알약 개수 분포
ax5 = plt.subplot(2, 3, 5)
sns.countplot(x=pills_per_image.values, palette='magma', ax=ax5)
ax5.set_title('[+] 이미지당 알약 개수 분포', fontproperties=fp, fontsize=15)
ax5.set_xlabel('알약 수'); ax5.set_ylabel('이미지 수')

# [6] 핵심 인사이트 요약
ax6 = plt.subplot(2, 3, 6)
ax6.axis('off')
summary_text = (
    "[데이터셋 핵심 인사이트]\n\n"
    f"• 총 이미지 수   : {len(images_df):,} 장\n"
    f"• 총 클래스 수   : {len(categories_df):,} 종\n"
    f"• 총 BBox 수     : {len(annotations_df):,} 개\n\n"
    f"• 평균 W/H 비율  : {annotations_df['aspect_ratio'].mean():.2f}\n"
    f"• 1장당 알약     : {pills_per_image.min()}~{pills_per_image.max()}개\n"
    f"• 면적 격차      : {max_area/min_area:.1f}배\n"
    f"• 클래스 불균형  : {class_counts.max()}/{class_counts.min()} = {class_counts.max()//class_counts.min()}배\n"
)
ax6.text(0.05, 0.55, summary_text, fontproperties=fp, fontsize=14,
         va='center', linespacing=2.0,
         bbox=dict(boxstyle='round,pad=0.8', facecolor='lightyellow', alpha=0.8))

plt.show()


## 3. 클래스별 상세 통계 리포트

In [ ]:
# Report 1: 전체 요약
total_summary = pd.DataFrame({
    '분석 지표': ['총 이미지 수', '총 클래스(알약 종류) 수', '총 BBox 수',
                  '이미지 1장당 평균 객체 수', '전체 평균 객체 면적(px²)'],
    '수치': [f"{len(images_df):,}", f"{len(categories_df):,}",
             f"{len(annotations_df):,}",
             f"{len(annotations_df)/len(images_df):.2f}",
             f"{annotations_df['area'].mean():,.2f}"]
})
print("="*50)
print("[Report 1] 전체 데이터셋 핵심 요약")
print("="*50)
display(total_summary)

# Report 2: 하위 10개 클래스
class_stats = annotations_df.groupby('class_name').agg(
    데이터개수=('id','count'),
    평균면적=('area','mean'),
    최소면적=('area','min'),
    최대면적=('area','max'),
    평균비율=('aspect_ratio','mean')
).reset_index().rename(columns={'class_name':'알약 이름'})
class_stats = class_stats.sort_values('데이터개수', ascending=False)

print("\n" + "="*50)
print("[Report 2] 🚨 롱테일 하위 10개 클래스 (증강 시급)")
print("="*50)
display(class_stats.tail(10).round(2))

# Report 3: BBox 크기 백분위수
print("\n" + "="*50)
print("[Report 3] 📐 BBox 크기 기술 통계 (Anchor Box 설계 기준)")
print("="*50)
area_desc = annotations_df['area'].describe(percentiles=[0.1,0.25,0.5,0.75,0.9,0.95])
display(pd.DataFrame(area_desc).T.round(2))


## 4. 픽셀 레벨 정밀 진단 (밝기 · 대비 · 평균 이미지)

In [ ]:
ORIG_IMG_DIR = os.path.join(BASE_DIR, 'train_images')

print("[!] 파일 경로 인덱싱 중...")
all_files = (glob.glob(os.path.join(BASE_DIR, '**', '*.png'), recursive=True) +
             glob.glob(os.path.join(BASE_DIR, '**', '*.jpg'), recursive=True) +
             glob.glob(os.path.join(BASE_DIR, '**', '*.JPG'), recursive=True))
path_map = {os.path.basename(f): f for f in all_files}
print(f"[OK] 총 {len(path_map):,}개 파일 매핑 완료")

def analyze_brightness_contrast(images_df, path_map, sample_n=500):
    brightness_list, contrast_list = [], []
    samples = images_df.sample(min(sample_n, len(images_df)), random_state=42)
    for _, row in samples.iterrows():
        fname = os.path.basename(row['file_name'])
        if fname in path_map:
            img = cv2.imread(path_map[fname], cv2.IMREAD_GRAYSCALE)
            if img is not None:
                brightness_list.append(np.mean(img))
                contrast_list.append(np.std(img))
    return brightness_list, contrast_list

def get_mean_images(annotations_df, images_df, path_map, target_classes, n_samples=30):
    mean_results = {}
    for cls_name in target_classes:
        cls_annos  = annotations_df[annotations_df['class_name'] == cls_name]
        cls_sample = cls_annos.sample(min(n_samples, len(cls_annos)), random_state=42)
        crops = []
        for _, anno in cls_sample.iterrows():
            img_info = images_df[images_df['id'] == anno['image_id']]
            if img_info.empty: continue
            fname = os.path.basename(img_info.iloc[0]['file_name'])
            if fname not in path_map: continue
            img = cv2.imread(path_map[fname])
            if img is None: continue
            x, y, w, h = map(int, anno['bbox'])
            ih, iw = img.shape[:2]
            crop = img[max(0,y):min(ih,y+h), max(0,x):min(iw,x+w)]
            if crop.size > 0:
                crops.append(cv2.resize(crop, (128, 128), interpolation=cv2.INTER_AREA))
        if crops:
            mean_results[cls_name] = np.mean(crops, axis=0).astype(np.uint8)
    return mean_results

print("\n[분석 1] 밝기/대비 스캔 중 (N=500)...")
brightness, contrast = analyze_brightness_contrast(images_df, path_map, 500)

print("[분석 2] Top5 클래스 평균 이미지 합성 중...")
top5 = annotations_df['class_name'].value_counts().head(5).index
mean_imgs = get_mean_images(annotations_df, images_df, path_map, top5)

# 대시보드
fig = plt.figure(figsize=(20, 10))
plt.subplots_adjust(hspace=0.4, wspace=0.3)

ax1 = plt.subplot(2, 2, 1)
sns.histplot(brightness, kde=True, color='darkorange', bins=30, ax=ax1)
ax1.axvline(np.mean(brightness), color='red', linestyle='--', linewidth=2,
            label=f'평균 {np.mean(brightness):.1f}')
ax1.set_title('■ 전체 이미지 밝기(Brightness) 분포', fontproperties=fp, fontsize=14)
ax1.legend(prop=fp)

ax2 = plt.subplot(2, 2, 2)
sns.histplot(contrast, kde=True, color='royalblue', bins=30, ax=ax2)
ax2.axvline(np.mean(contrast), color='red', linestyle='--', linewidth=2,
            label=f'평균 {np.mean(contrast):.1f}')
ax2.set_title('■ 전체 이미지 대비(Contrast) 분포', fontproperties=fp, fontsize=14)
ax2.legend(prop=fp)

for i, (name, img) in enumerate(mean_imgs.items()):
    ax = plt.subplot(2, 5, 6 + i)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(name[:12], fontproperties=fp, fontsize=10)
    ax.axis('off')

plt.suptitle('■ [Health Eat] 픽셀 레벨 정밀 분석', fontproperties=fp,
             fontsize=20, fontweight='bold', y=0.98)
plt.show()

# 수치 리포트
print("\n[픽셀 진단 요약]")
display(pd.DataFrame({
    '평균 밝기 (0~255)': [round(np.mean(brightness), 2)],
    '밝기 표준편차':      [round(np.std(brightness), 2)],
    '평균 대비':          [round(np.mean(contrast), 2)],
    '최소 밝기':          [round(np.min(brightness), 2)],
    '최대 밝기':          [round(np.max(brightness), 2)],
}))
print("\n💡 [인사이트] 평균 대비가 18 수준 → CLAHE 전처리 필수 근거")


## 5. 📸 랜덤 샘플 20장 확인
> BBox가 정상적으로 붙어있는지 육안으로 확인

In [ ]:
from src.preprocessing.viz_utils import show_samples
show_samples(ORIG_IMG_DIR, TRAIN_JSON, n=20, title="원본 데이터 샘플 (BBox 포함)")


## 6. EDA 결과 리포트 및 전처리 전략

### 1. 데이터 정밀 분석 결과 요약

**1.1 클래스 불균형 — 🚨 Critical Issue**
- 최다 클래스: 기넥신에프정 (514개) vs 최소 클래스: 브린텔릭스정 (7개)
- 상하위 클래스 비율 차이 약 **73배** → 일반 학습 시 하위 클래스 mAP ≈ 0 예상

**1.2 기하학적 특징**
- 면적 편차: 약 **14.7배** → Letterbox 전처리 필수
- Aspect Ratio 평균 ≈ 0.99 → 대부분 정사각형에 가까운 BBox
- 위치 히트맵: 알약이 화면 중앙부에 집중 → 구석 탐지력 약화 가능성

**1.3 광학 특징**
- 평균 대비(Contrast) ≈ 18 → 각인이 배경에 묻힘 → **CLAHE 필수**
- 클래스별 평균 이미지에서 Ghosting 현상 → 알약이 무작위 각도로 회전되어 있음

---

### 2. 전처리 전략 결정 근거

| 전처리 | 근거 | 파라미터 |
|---|---|---|
| **Stratified Split** | 73x 클래스 불균형 → 랜덤 split 시 소수 클래스 val 누락 | test_size=0.1, random_state=42 |
| **Copy-Paste 증강** | 소수 클래스 7~11샘플 → 모델 학습 불가 수준 | threshold=50, aug_count=500 |
| **Letterbox 800×800** | 14.7x 크기 분산 → 동일 해상도 정규화 필요 | target_size=800, pad=114 |
| **CLAHE** | 평균 대비 18 → 각인 식별력 저하 | clipLimit=2.0, tileGridSize=(8,8) |


## ✅ NB01 완료
> 다음 단계: NB02 - Stratified Split + Copy-Paste 증강